# 02 — KPI Testing

Validate dashboard KPI outputs against `docs/kpi_definitions.md` and the SQL warehouse layers.

## Testing Dimensions

1. **Grain Validation**: confirm each gold/dashboard table has the expected row grain and no accidental duplication.
2. **Metric Validation**: recalculate KPI values from gold tables and compare them with dashboard marts.
3. **Logic Validation**: verify business rules such as delivered-only filters, `customer_unique_id` retention logic, installment rules, and delivery-delay eligibility.

In [9]:
from pathlib import Path
import os
import importlib
import subprocess
import sys
import getpass
from urllib.parse import quote_plus


def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


ensure_pkg("sqlalchemy", "sqlalchemy")
ensure_pkg("psycopg2", "psycopg2-binary")

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

ROOT = Path("..").resolve()
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "ecommerce")
DB_USER = os.getenv("POSTGRES_USER") or getpass.getuser()
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "")

password_part = quote_plus(DB_PASSWORD)
conn_str = f"postgresql+psycopg2://{DB_USER}:{password_part}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)


def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(text(sql), engine)


def show_check(sql: str) -> pd.DataFrame:
    df = q(sql)
    return df

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print(f"Connected target: {DB_HOST}:{DB_PORT}/{DB_NAME} as {DB_USER}")

Connected target: localhost:5432/ecommerce as sherrywang


## 1) Grain Validation

Purpose: make sure every table is at the intended grain before using it for KPI calculations. This catches row multiplication issues, especially joins involving order items, payments, and reviews.

In [2]:
grain_checks = show_check("""
WITH checks AS (
    SELECT
        'gold_fact_orders' AS table_name,
        'one row per order_id' AS expected_grain,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_rows
    FROM gold_fact_orders

    UNION ALL

    SELECT
        'gold_fact_order_items',
        'one row per (order_id, order_item_id)',
        COUNT(*),
        COUNT(DISTINCT (order_id, order_item_id)),
        COUNT(*) - COUNT(DISTINCT (order_id, order_item_id))
    FROM gold_fact_order_items

    UNION ALL

    SELECT
        'gold_fact_order_payments',
        'one row per (order_id, payment_sequential)',
        COUNT(*),
        COUNT(DISTINCT (order_id, payment_sequential)),
        COUNT(*) - COUNT(DISTINCT (order_id, payment_sequential))
    FROM gold_fact_order_payments

    UNION ALL

    SELECT
        'gold_fact_order_reviews',
        'one row per (review_id, order_id)',
        COUNT(*),
        COUNT(DISTINCT (review_id, order_id)),
        COUNT(*) - COUNT(DISTINCT (review_id, order_id))
    FROM gold_fact_order_reviews

    UNION ALL

    SELECT
        'gold_customer_delivered_orders',
        'one row per delivered order_id with customer_unique_id',
        COUNT(*),
        COUNT(DISTINCT order_id),
        COUNT(*) - COUNT(DISTINCT order_id)
    FROM gold_customer_delivered_orders

    UNION ALL

    SELECT
        'dash_kpi_monthly',
        'one row per report_month',
        COUNT(*),
        COUNT(DISTINCT report_month),
        COUNT(*) - COUNT(DISTINCT report_month)
    FROM dash_kpi_monthly

    UNION ALL

    SELECT
        'dash_retention_snapshot',
        'one row per metric_code at dataset as-of',
        COUNT(*),
        COUNT(DISTINCT metric_code),
        COUNT(*) - COUNT(DISTINCT metric_code)
    FROM dash_retention_snapshot

    UNION ALL

    SELECT
        'dash_retention_monthly',
        'one row per (report_month, metric_code)',
        COUNT(*),
        COUNT(DISTINCT (report_month, metric_code)),
        COUNT(*) - COUNT(DISTINCT (report_month, metric_code))
    FROM dash_retention_monthly
)
SELECT
    table_name,
    expected_grain,
    total_rows,
    distinct_keys,
    duplicate_rows,
    CASE WHEN duplicate_rows = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks
ORDER BY table_name;
""")

grain_checks

,table_name,expected_grain,total_rows,distinct_keys,duplicate_rows,status
0,dash_kpi_monthly,one row per report_month,23,23,0,PASS
1,dash_retention_monthly,"one row per (report_month, metric_code)",115,115,0,PASS
2,dash_retention_snapshot,one row per metric_code at dataset as-of,5,5,0,PASS
3,gold_customer_delivered_orders,one row per delivered order_id with customer_u...,96478,96478,0,PASS
4,gold_fact_order_items,"one row per (order_id, order_item_id)",112650,112650,0,PASS
5,gold_fact_order_payments,"one row per (order_id, payment_sequential)",103886,103886,0,PASS
6,gold_fact_order_reviews,"one row per (review_id, order_id)",96361,96361,0,PASS
7,gold_fact_orders,one row per order_id,99441,99441,0,PASS


In [3]:
retention_grain_checks = show_check("""
WITH expected_codes AS (
    SELECT * FROM (VALUES
        ('lifetime_repeat_purchase'),
        ('last_3m_returning_customer'),
        ('last_6m_returning_customer'),
        ('last_3m_repeat_purchase'),
        ('last_6m_repeat_purchase')
    ) AS t(metric_code)
),
snapshot_missing AS (
    SELECT e.metric_code
    FROM expected_codes e
    LEFT JOIN dash_retention_snapshot s
        ON e.metric_code = s.metric_code
    WHERE s.metric_code IS NULL
),
monthly_code_counts AS (
    SELECT
        report_month,
        COUNT(DISTINCT metric_code) AS metric_code_count
    FROM dash_retention_monthly
    GROUP BY report_month
)
SELECT
    'dash_retention_snapshot metric codes' AS check_name,
    (SELECT COUNT(*) FROM expected_codes) AS expected_count,
    (SELECT COUNT(DISTINCT metric_code) FROM dash_retention_snapshot) AS actual_count,
    (SELECT COUNT(*) FROM snapshot_missing) AS missing_count,
    CASE
        WHEN (SELECT COUNT(*) FROM snapshot_missing) = 0
         AND (SELECT COUNT(DISTINCT metric_code) FROM dash_retention_snapshot) = 5
        THEN 'PASS' ELSE 'FAIL'
    END AS status

UNION ALL

SELECT
    'dash_retention_monthly has 5 metric codes per month',
    5,
    MIN(metric_code_count),
    COUNT(*) FILTER (WHERE metric_code_count <> 5),
    CASE WHEN COUNT(*) FILTER (WHERE metric_code_count <> 5) = 0 THEN 'PASS' ELSE 'FAIL' END
FROM monthly_code_counts;
""")

retention_grain_checks

,check_name,expected_count,actual_count,missing_count,status
0,dash_retention_snapshot metric codes,5,5,0,PASS
1,dash_retention_monthly has 5 metric codes per ...,5,5,0,PASS


## 2) Metric Validation

Purpose: recalculate dashboard metrics directly from gold facts and compare with dashboard mart values. Differences should be zero or within rounding tolerance.

In [4]:
monthly_metric_validation = show_check("""
WITH recalculated AS (
    SELECT
        m.report_month,
        COALESCE(i.gmv, 0) AS gmv,
        COALESCE(i.total_orders, 0) AS total_orders,
        CASE WHEN COALESCE(i.total_orders, 0) > 0 THEN COALESCE(i.gmv, 0) / i.total_orders END AS aov,
        COALESCE(p.actual_payment, 0) AS actual_payment,
        COALESCE(o.delivery_delay_denominator, 0) AS delivery_delay_denominator,
        COALESCE(o.delayed_orders, 0) AS delayed_orders,
        CASE WHEN COALESCE(o.delivery_delay_denominator, 0) > 0 THEN o.delayed_orders::numeric / o.delivery_delay_denominator END AS delivery_delay_rate,
        rv.avg_review_score,
        COALESCE(o.installment_orders, 0) AS installment_orders,
        CASE WHEN COALESCE(o.delivered_orders, 0) > 0 THEN o.installment_orders::numeric / o.delivered_orders END AS installment_usage_rate
    FROM (
        SELECT DISTINCT order_purchase_month AS report_month
        FROM gold_fact_orders
        WHERE is_delivered AND order_purchase_month IS NOT NULL
    ) m
    LEFT JOIN (
        SELECT order_purchase_month AS report_month,
               SUM(line_gmv) AS gmv,
               COUNT(DISTINCT order_id) AS total_orders
        FROM gold_fact_order_items
        WHERE is_delivered AND order_purchase_month IS NOT NULL
        GROUP BY order_purchase_month
    ) i ON m.report_month = i.report_month
    LEFT JOIN (
        SELECT order_purchase_month AS report_month,
               SUM(payment_value) AS actual_payment
        FROM gold_fact_order_payments
        WHERE is_delivered AND order_purchase_month IS NOT NULL
        GROUP BY order_purchase_month
    ) p ON m.report_month = p.report_month
    LEFT JOIN (
        SELECT order_purchase_month AS report_month,
               COUNT(*) FILTER (WHERE is_delivered) AS delivered_orders,
               COUNT(*) FILTER (WHERE is_valid_for_delivery_delay_kpi) AS delivery_delay_denominator,
               COUNT(*) FILTER (WHERE is_delayed) AS delayed_orders,
               COUNT(*) FILTER (WHERE is_delivered AND uses_installments) AS installment_orders
        FROM gold_fact_orders
        WHERE order_purchase_month IS NOT NULL
        GROUP BY order_purchase_month
    ) o ON m.report_month = o.report_month
    LEFT JOIN (
        SELECT order_purchase_month AS report_month,
               AVG(review_score) AS avg_review_score
        FROM gold_fact_order_reviews
        WHERE order_purchase_month IS NOT NULL
        GROUP BY order_purchase_month
    ) rv ON m.report_month = rv.report_month
),
diffs AS (
    SELECT
        d.report_month,
        ABS(d.gmv - r.gmv) AS gmv_diff,
        ABS(d.total_orders - r.total_orders) AS total_orders_diff,
        ABS(d.aov - r.aov) AS aov_diff,
        ABS(d.actual_payment - r.actual_payment) AS actual_payment_diff,
        ABS(d.delivery_delay_rate - r.delivery_delay_rate) AS delivery_delay_rate_diff,
        ABS(d.avg_review_score - r.avg_review_score) AS avg_review_score_diff,
        ABS(d.installment_usage_rate - r.installment_usage_rate) AS installment_usage_rate_diff
    FROM dash_kpi_monthly d
    INNER JOIN recalculated r
        ON d.report_month = r.report_month
)
SELECT
    COUNT(*) AS months_checked,
    MAX(gmv_diff) AS max_gmv_diff,
    MAX(total_orders_diff) AS max_total_orders_diff,
    MAX(aov_diff) AS max_aov_diff,
    MAX(actual_payment_diff) AS max_actual_payment_diff,
    MAX(delivery_delay_rate_diff) AS max_delivery_delay_rate_diff,
    MAX(avg_review_score_diff) AS max_avg_review_score_diff,
    MAX(installment_usage_rate_diff) AS max_installment_usage_rate_diff,
    CASE
        WHEN MAX(gmv_diff) <= 0.01
         AND MAX(total_orders_diff) = 0
         AND MAX(aov_diff) <= 0.000001
         AND MAX(actual_payment_diff) <= 0.01
         AND MAX(delivery_delay_rate_diff) <= 0.000001
         AND MAX(avg_review_score_diff) <= 0.000001
         AND MAX(installment_usage_rate_diff) <= 0.000001
        THEN 'PASS' ELSE 'FAIL'
    END AS status
FROM diffs;
""")

monthly_metric_validation

,months_checked,max_gmv_diff,max_total_orders_diff,max_aov_diff,max_actual_payment_diff,max_delivery_delay_rate_diff,max_avg_review_score_diff,max_installment_usage_rate_diff,status
0,23,0.0,0,0.0,0.0,0.0,0.0,0.0,PASS


In [5]:
retention_metric_validation = show_check("""
WITH params AS (
    SELECT
        as_of_timestamp,
        as_of_timestamp - INTERVAL '3 months' AS window_3m_start,
        as_of_timestamp - INTERVAL '6 months' AS window_6m_start
    FROM gold_dataset_dates
),
lifetime AS (
    SELECT
        'lifetime_repeat_purchase' AS metric_code,
        COUNT(*) AS total_customers,
        COUNT(*) FILTER (WHERE delivered_order_count > 1) AS returning_customers
    FROM (
        SELECT customer_unique_id, COUNT(DISTINCT order_id) AS delivered_order_count
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp <= p.as_of_timestamp
        GROUP BY customer_unique_id
    ) t
),
last_3m_returning AS (
    SELECT
        'last_3m_returning_customer' AS metric_code,
        COUNT(DISTINCT iw.customer_unique_id) AS total_customers,
        COUNT(DISTINCT bw.customer_unique_id) AS returning_customers
    FROM (
        SELECT DISTINCT customer_unique_id
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp > p.window_3m_start
          AND g.order_purchase_timestamp <= p.as_of_timestamp
    ) iw
    LEFT JOIN (
        SELECT DISTINCT customer_unique_id
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp <= p.window_3m_start
    ) bw ON iw.customer_unique_id = bw.customer_unique_id
),
last_6m_returning AS (
    SELECT
        'last_6m_returning_customer' AS metric_code,
        COUNT(DISTINCT iw.customer_unique_id) AS total_customers,
        COUNT(DISTINCT bw.customer_unique_id) AS returning_customers
    FROM (
        SELECT DISTINCT customer_unique_id
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp > p.window_6m_start
          AND g.order_purchase_timestamp <= p.as_of_timestamp
    ) iw
    LEFT JOIN (
        SELECT DISTINCT customer_unique_id
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp <= p.window_6m_start
    ) bw ON iw.customer_unique_id = bw.customer_unique_id
),
last_3m_repeat AS (
    SELECT
        'last_3m_repeat_purchase' AS metric_code,
        COUNT(*) AS total_customers,
        COUNT(*) FILTER (WHERE window_order_count > 1) AS returning_customers
    FROM (
        SELECT customer_unique_id, COUNT(DISTINCT order_id) AS window_order_count
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp > p.window_3m_start
          AND g.order_purchase_timestamp <= p.as_of_timestamp
        GROUP BY customer_unique_id
    ) t
),
last_6m_repeat AS (
    SELECT
        'last_6m_repeat_purchase' AS metric_code,
        COUNT(*) AS total_customers,
        COUNT(*) FILTER (WHERE window_order_count > 1) AS returning_customers
    FROM (
        SELECT customer_unique_id, COUNT(DISTINCT order_id) AS window_order_count
        FROM gold_customer_delivered_orders g
        CROSS JOIN params p
        WHERE g.order_purchase_timestamp > p.window_6m_start
          AND g.order_purchase_timestamp <= p.as_of_timestamp
        GROUP BY customer_unique_id
    ) t
),
recalculated AS (
    SELECT * FROM lifetime
    UNION ALL SELECT * FROM last_3m_returning
    UNION ALL SELECT * FROM last_6m_returning
    UNION ALL SELECT * FROM last_3m_repeat
    UNION ALL SELECT * FROM last_6m_repeat
),
diffs AS (
    SELECT
        d.metric_code,
        d.total_customers AS dash_total_customers,
        r.total_customers AS recalculated_total_customers,
        d.returning_customers AS dash_returning_customers,
        r.returning_customers AS recalculated_returning_customers,
        ABS(d.repeat_purchase_rate - (r.returning_customers::numeric / NULLIF(r.total_customers, 0))) AS rate_diff
    FROM dash_retention_snapshot d
    INNER JOIN recalculated r
        ON d.metric_code = r.metric_code
)
SELECT
    metric_code,
    dash_total_customers,
    recalculated_total_customers,
    dash_returning_customers,
    recalculated_returning_customers,
    rate_diff,
    CASE
        WHEN dash_total_customers = recalculated_total_customers
         AND dash_returning_customers = recalculated_returning_customers
         AND rate_diff <= 0.000001
        THEN 'PASS' ELSE 'FAIL'
    END AS status
FROM diffs
ORDER BY metric_code;
""")

retention_metric_validation

,metric_code,dash_total_customers,recalculated_total_customers,dash_returning_customers,recalculated_returning_customers,rate_diff,status
0,last_3m_repeat_purchase,18728,18728,214,214,0.0,PASS
1,last_3m_returning_customer,18728,18728,423,423,0.0,PASS
2,last_6m_repeat_purchase,38645,38645,637,637,0.0,PASS
3,last_6m_returning_customer,38645,38645,651,651,0.0,PASS
4,lifetime_repeat_purchase,93358,93358,2801,2801,0.0,PASS


In [6]:
category_metric_validation = show_check("""
WITH recalculated AS (
    SELECT
        order_purchase_month AS report_month,
        product_category_name_english AS category,
        SUM(line_gmv) AS category_gmv,
        COUNT(DISTINCT order_id) AS category_order_count
    FROM gold_fact_order_items
    WHERE is_delivered
      AND order_purchase_month IS NOT NULL
    GROUP BY order_purchase_month, product_category_name_english
),
diffs AS (
    SELECT
        d.report_month,
        d.category,
        ABS(d.category_gmv - r.category_gmv) AS category_gmv_diff,
        ABS(d.category_order_count - r.category_order_count) AS category_order_count_diff
    FROM dash_category_monthly d
    INNER JOIN recalculated r
        ON d.report_month = r.report_month
       AND d.category = r.category
)
SELECT
    COUNT(*) AS category_months_checked,
    MAX(category_gmv_diff) AS max_category_gmv_diff,
    MAX(category_order_count_diff) AS max_category_order_count_diff,
    CASE
        WHEN MAX(category_gmv_diff) <= 0.01
         AND MAX(category_order_count_diff) = 0
        THEN 'PASS' ELSE 'FAIL'
    END AS status
FROM diffs;
""")

category_metric_validation

,category_months_checked,max_category_gmv_diff,max_category_order_count_diff,status
0,1243,0.0,0,PASS


## 3) Logic Validation

Purpose: test whether business logic matches KPI definitions, not just whether numbers add up.

In [7]:
logic_checks = show_check("""
SELECT
    'delivered-only sales facts' AS check_name,
    COUNT(*) AS issue_count,
    CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
    'dash sales KPIs should come from delivered orders only' AS rationale
FROM gold_fact_order_items
WHERE is_delivered = false
  AND order_id IN (SELECT DISTINCT order_id FROM gold_fact_order_items WHERE is_delivered)

UNION ALL

SELECT
    'retention uses customer_unique_id, not customer_id',
    COUNT(*) FILTER (WHERE delivered_orders_by_customer_id > 1),
    CASE WHEN COUNT(*) FILTER (WHERE delivered_orders_by_customer_id > 1) = 0 THEN 'PASS' ELSE 'FAIL' END,
    'Olist customer_id is order-level; repeat behavior must use customer_unique_id'
FROM (
    SELECT customer_id, COUNT(DISTINCT order_id) AS delivered_orders_by_customer_id
    FROM gold_customer_delivered_orders
    GROUP BY customer_id
) t

UNION ALL

SELECT
    'customer_unique_id has repeat buyers',
    COUNT(*) FILTER (WHERE delivered_orders_by_unique_id > 1),
    CASE WHEN COUNT(*) FILTER (WHERE delivered_orders_by_unique_id > 1) > 0 THEN 'PASS' ELSE 'FAIL' END,
    'Repeat purchase metrics should be non-zero when using customer_unique_id'
FROM (
    SELECT customer_unique_id, COUNT(DISTINCT order_id) AS delivered_orders_by_unique_id
    FROM gold_customer_delivered_orders
    GROUP BY customer_unique_id
) t

UNION ALL

SELECT
    'installment numerator uses payment_installments > 1',
    COUNT(*) FILTER (WHERE uses_installments AND max_installments <= 1),
    CASE WHEN COUNT(*) FILTER (WHERE uses_installments AND max_installments <= 1) = 0 THEN 'PASS' ELSE 'FAIL' END,
    'Installment orders must have at least one payment row with payment_installments > 1'
FROM (
    SELECT
        o.order_id,
        o.uses_installments,
        MAX(p.payment_installments) AS max_installments
    FROM gold_fact_orders o
    LEFT JOIN gold_fact_order_payments p
        ON o.order_id = p.order_id
    WHERE o.is_delivered
    GROUP BY o.order_id, o.uses_installments
) t

UNION ALL

SELECT
    'payment_installments = 0 not counted as installment',
    COUNT(*) FILTER (WHERE payment_installments = 0 AND is_installment_payment),
    CASE WHEN COUNT(*) FILTER (WHERE payment_installments = 0 AND is_installment_payment) = 0 THEN 'PASS' ELSE 'FAIL' END,
    'Data quirk value 0 should not count as installment usage'
FROM gold_fact_order_payments

UNION ALL

SELECT
    'delivery delay denominator excludes inconsistent delivery records',
    COUNT(*) FILTER (WHERE is_valid_for_delivery_delay_kpi AND is_delivery_inconsistency),
    CASE WHEN COUNT(*) FILTER (WHERE is_valid_for_delivery_delay_kpi AND is_delivery_inconsistency) = 0 THEN 'PASS' ELSE 'FAIL' END,
    'Delay KPI requires delivered + valid delivery dates + no inconsistency flag'
FROM gold_fact_orders

UNION ALL

SELECT
    'review fact is delivered orders only',
    COUNT(*) FILTER (WHERE o.order_status <> 'delivered'),
    CASE WHEN COUNT(*) FILTER (WHERE o.order_status <> 'delivered') = 0 THEN 'PASS' ELSE 'FAIL' END,
    'Average review score KPI joins reviews to delivered orders only'
FROM gold_fact_order_reviews r
INNER JOIN silver_olist_orders_dataset o
    ON r.order_id = o.order_id

UNION ALL

SELECT
    'reviews do not inflate gold_fact_orders grain',
    COUNT(*) - COUNT(DISTINCT order_id),
    CASE WHEN COUNT(*) - COUNT(DISTINCT order_id) = 0 THEN 'PASS' ELSE 'FAIL' END,
    'gold_fact_orders aggregates reviews before joining because order_id is not unique in reviews'
FROM gold_fact_orders;
""")

logic_checks

,check_name,issue_count,status,rationale
0,payment_installments = 0 not counted as instal...,0,PASS,Data quirk value 0 should not count as install...
1,delivery delay denominator excludes inconsiste...,0,PASS,Delay KPI requires delivered + valid delivery ...
2,reviews do not inflate gold_fact_orders grain,0,PASS,gold_fact_orders aggregates reviews before joi...
3,installment numerator uses payment_installment...,0,PASS,Installment orders must have at least one paym...
4,review fact is delivered orders only,0,PASS,Average review score KPI joins reviews to deli...
5,delivered-only sales facts,0,PASS,dash sales KPIs should come from delivered ord...
6,"retention uses customer_unique_id, not custome...",0,PASS,Olist customer_id is order-level; repeat behav...
7,customer_unique_id has repeat buyers,2801,PASS,Repeat purchase metrics should be non-zero whe...


In [8]:
window_logic_checks = show_check("""
WITH snapshot AS (
    SELECT metric_code, total_customers, returning_customers, repeat_purchase_rate
    FROM dash_retention_snapshot
),
comparisons AS (
    SELECT
        'last_3m_returning_customer >= last_3m_repeat_purchase usually expected' AS check_name,
        (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_3m_returning_customer') AS left_value,
        (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_3m_repeat_purchase') AS right_value,
        CASE
            WHEN (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_3m_returning_customer')
              >= (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_3m_repeat_purchase')
            THEN 'PASS' ELSE 'REVIEW'
        END AS status,
        'Returning customer = bought in window and before window; repeat purchase = >1 orders inside window' AS rationale

    UNION ALL

    SELECT
        'last_6m_returning_customer >= last_6m_repeat_purchase usually expected',
        (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_6m_returning_customer'),
        (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_6m_repeat_purchase'),
        CASE
            WHEN (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_6m_returning_customer')
              >= (SELECT returning_customers FROM snapshot WHERE metric_code = 'last_6m_repeat_purchase')
            THEN 'PASS' ELSE 'REVIEW'
        END,
        'Both are valid but answer different retention questions'
)
SELECT * FROM comparisons;
""")

window_logic_checks

,check_name,left_value,right_value,status,rationale
0,last_3m_returning_customer >= last_3m_repeat_p...,423,214,PASS,Returning customer = bought in window and befo...
1,last_6m_returning_customer >= last_6m_repeat_p...,651,637,PASS,Both are valid but answer different retention ...


## Test Summary

Run all cells above. Any `FAIL` should be investigated before using the KPI table in reporting. `REVIEW` means the check is not strictly invalid, but deserves business interpretation.